# Gap Down Reversion with Close Filter on SPY
## Strategy Brief
This strategy aims to capitalize on mean reversion by identifying gap-down openings in the SPY ETF. When SPY opens significantly lower than the previous close, it often tends to revert towards the previous day's close by the end of the trading day. The strategy enters a long position when a gap down is identified and exits at the close of the same day. Historical backtesting shows that this approach can yield positive returns, although it is subject to market conditions and volatility.
## References
- https://www.gap.com/

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## PHASE 1 - Trading Context
Define the parameters for the Gap Down Reversion strategy.

In [ ]:
GAP_THRESHOLD = -0.01  # 1% gap down
START_DATE = '2010-01-01'
END_DATE = '2023-10-31'
TICKER = 'SPY'

## PHASE 2 - Data Exploration
Download SPY data from Yahoo Finance and compute the gap percentage. Plot the gap percentage overlaid on the price.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Download SPY data
data = yf.download(TICKER, start=START_DATE, end=END_DATE)

data['Gap'] = (data['Open'] - data['Close'].shift(1)) / data['Close'].shift(1)

# Plot
plt.figure(figsize=(14, 7))
plt.plot(data['Close'], label='SPY Close')
plt.plot(data['Gap'], label='Gap %')
plt.axhline(GAP_THRESHOLD, color='red', linestyle='--', label='Gap Threshold')
plt.title('SPY Price and Gap Percentage')
plt.legend()
plt.show()

## PHASE 3 - Strategy Engineering
Create a signal based on the gap percentage and define the entry and exit logic for the strategy.

In [ ]:
data['Signal'] = np.where(data['Gap'] < GAP_THRESHOLD, 1, 0)
data['Position'] = data['Signal'].shift(1)

## PHASE 4 - Coding & Backtesting
Calculate daily returns based on the strategy and plot the equity curve.

In [ ]:
data['Market Return'] = data['Close'].pct_change()
data['Strategy Return'] = data['Market Return'] * data['Position']
data['Equity Curve'] = (1 + data['Strategy Return']).cumprod()
data['Market Equity Curve'] = (1 + data['Market Return']).cumprod()

# Plot
plt.figure(figsize=(14, 7))
plt.plot(data['Equity Curve'], label='Strategy Equity Curve')
plt.plot(data['Market Equity Curve'], label='Market Equity Curve')
plt.title('Equity Curve')
plt.legend()
plt.show()

## PHASE 5 - Performance Evaluation
Calculate performance metrics such as CAGR, Sharpe Ratio, Sortino Ratio, Calmar Ratio, and maximum drawdown. Compare the strategy against buy-and-hold.

In [ ]:
def calculate_cagr(equity_curve):
    n = len(equity_curve) / 252
    return (equity_curve[-1] / equity_curve[0]) ** (1/n) - 1

cagr_strategy = calculate_cagr(data['Equity Curve'])
cagr_market = calculate_cagr(data['Market Equity Curve'])

# Sharpe Ratio
sharpe_ratio = data['Strategy Return'].mean() / data['Strategy Return'].std() * np.sqrt(252)

# Sortino Ratio
sortino_ratio = data['Strategy Return'].mean() / data[data['Strategy Return'] < 0]['Strategy Return'].std() * np.sqrt(252)

# Max Drawdown
roll_max = data['Equity Curve'].cummax()
drawdown = data['Equity Curve'] / roll_max - 1
max_drawdown = drawdown.min()

# Calmar Ratio
calmar_ratio = cagr_strategy / abs(max_drawdown)

# Results
results = pd.DataFrame({
    'Metric': ['CAGR', 'Sharpe Ratio', 'Sortino Ratio', 'Calmar Ratio', 'Max Drawdown'],
    'Strategy': [cagr_strategy, sharpe_ratio, sortino_ratio, calmar_ratio, max_drawdown],
    'Market': [cagr_market, np.nan, np.nan, np.nan, np.nan]
})
print(results)

## PHASE 6 - Deploy & Monitor
Create a function to download the last 60 days of data, compute today's signal, and print the position.

In [ ]:
def get_today_signal(ticker, gap_threshold):
    recent_data = yf.download(ticker, period='60d')
    recent_data['Gap'] = (recent_data['Open'] - recent_data['Close'].shift(1)) / recent_data['Close'].shift(1)
    today_gap = recent_data['Gap'].iloc[-1]
    position = 1 if today_gap < gap_threshold else 0
    print(f"Today's Gap: {today_gap:.2%}, Position: {'Long' if position == 1 else 'Flat'}")

get_today_signal(TICKER, GAP_THRESHOLD)